# Llama 3.2 1B — Unsloth + LoRA + 4-bit Quantization Setup

**Purpose:** Infrastructure scaffold for ARL drone detection LLM fine-tuning.  
**Status:** Runs end-to-end on dummy data. Swap in real `snapshot_log.csv` data when teammates finish.

**Stack:**
- `unsloth` — fast loader, handles quant + LoRA setup in one call
- `4-bit quantization` — fits Llama 3.2 1B in T4's 16GB VRAM
- `LoRA` — only trains small adapter matrices, not the full 1B weights
- `SFTTrainer` (TRL) — supervised fine-tuning training loop
- `HuggingFace Hub` — model source (needs HF_TOKEN with Llama access)

## Cell 1 — Install dependencies

In [ ]:
# Unsloth has a specific install order — do NOT reorder these
# This takes ~3-5 minutes on first run
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q huggingface_hub datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 151.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 24.3 MB/s eta 0:00:00


## Cell 2 — HuggingFace login

In [ ]:
import os

def get_secret(key_name):
    """Fetch a secret from Colab, Kaggle, or env — whichever is available."""
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except Exception:
        pass
    return os.getenv(key_name)

hf_token = get_secret("HF_TOKEN")
assert hf_token, "HF_TOKEN not found — add it in Kaggle > Add-ons > Secrets"

# Log in so vLLM can pull the gated model
from huggingface_hub import login
login(token=hf_token)

## Cell 3 — Load model with Unsloth (quant + LoRA in one call)

In [ ]:
from unsloth import FastLanguageModel
import torch

# ── Config ───────────────────────────────────────────────
MODEL_NAME     = "meta-llama/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH = 4096   # max tokens per training example
DTYPE          = torch.bfloat16   # None = auto-detect (float16 on T4)
LOAD_IN_4BIT   = True   # 4-bit quantization to fit in T4 VRAM

# ── Load base model (Unsloth handles quant automatically) ─
# This is the key difference vs plain HuggingFace:
# Instead of AutoModelForCausalLM.from_pretrained() + BitsAndBytesConfig,
# Unsloth does it all in one fast call.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
    token          = os.environ.get("HF_TOKEN"),
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Device: {next(model.parameters()).device}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model loaded: meta-llama/Llama-3.2-1B-Instruct
Device: cuda:0


## Cell 4 — Attach LoRA adapters

In [ ]:
# ── LoRA Config ───────────────────────────────────────────
# r=8: LoRA rank — how big the adapter matrices are.
#      Higher = more capacity but slower. 8 is a good default for small tasks.
# target_modules: which attention layers to inject adapters into.
# lora_alpha: scaling factor. Rule of thumb: set to 2x the rank.
# lora_dropout=0: Unsloth is optimized for dropout=0 (faster kernels).
# bias='none': don't train bias terms (standard for LoRA).

model = FastLanguageModel.get_peft_model(
    model,
    r                  = 8,
    target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
    lora_alpha         = 16,
    lora_dropout       = 0,
    bias               = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth's memory optimization
    random_state       = 42,
)

# Show how many parameters are actually being trained
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
# Expect: ~1-2% of total params — that's the whole point of LoRA

Unsloth 2026.6.9 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Trainable params: 5,636,096 / 780,077,056 (0.72%)


## Cell 5 — Build dummy dataset

These are placeholder instruction-response pairs that teach the model the vocabulary of your system.
**When teammates finish:** replace this with real operator queries derived from `snapshot_log.csv`.

In [ ]:
from datasets import load_dataset

INSTRUCTION = """You are the AI assistant for an airport counter-drone threat-detection system. Operators, responders, and members of the public ask you what is happening — sometimes formally, sometimes anxiously, sometimes casually — and you give them a clear, faithful answer based on the current sensor readings.

THE SENSOR NETWORK
The site is covered by 23 sensors arranged in a ring around the airfield, indexed 0 through 22. Each sensor carries three independent detectors:
- Radio Frequency (RF): classifies radio emissions as friendly or threat.
- Audio: classifies sound as Mambo drone (threat), Bebop drone (threat), or Background noise (no threat).
- Visual: a camera score from 0 (no drone visible) to 1 (drone clearly visible).

A fusion model combines every sensor's readings into a single system-wide threat estimate. You receive that estimate at the top of each snapshot, then the per-sensor breakdown.

SENSOR GROUPINGS BY REGION
When the user asks about an area, use these named groups:
- Quadrants:
  - First quadrant:  sensors 11-16
  - Second quadrant: sensors 5-11
  - Third quadrant:  sensors 0-5
  - Fourth quadrant: sensors 16-22
- Hemispheres:
  - North: sensors 5-16
  - South: sensors 0-5 and 16-22
  - East:  sensors 11-22
  - West:  sensors 0-11

HOW TO ANSWER
- Ground every claim in the snapshot. NEVER invent numbers, sensors, or readings the snapshot does not show.
- Match the user's voice.
- For lay users, do NOT use technical words like "logit", "softmax", "modality", or "confidence vector".
- Do not dump the entire snapshot. Focus on what the user asked.
- When several sensors or modalities agree, say so.
- When asked about direction or area, use the named groupings above.
- Do not offer to display images, play audio, pull spectrograms, or take any action outside answering the question.
- Be honest about uncertainty. If a reading is borderline (around 50%), say so.
- Keep responses appropriately brief. One to four sentences is usually right."""

dataset = load_dataset(
    "JamesResearch1216/threat-detection-responses-10k",
    split="train",
)

rows = []
for example in dataset:
    user_content = (
        f"USER QUESTION:\n{example['query']}\n\n"
        f"CURRENT SENSOR READINGS:\n{example['snapshot_string']}"
    )

    messages = [
        {"role": "system",    "content": INSTRUCTION},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": example["response"]},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    rows.append({"text": text})

from datasets import Dataset
dataset = Dataset.from_list(rows)

print(f"Dataset size: {len(dataset)}")
print("\nSample formatted example:")
print(dataset[0]["text"][:1000])

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

train-00090.parquet:   0%|          | 0.00/138k [00:00<?, ?B/s]

train-00110.parquet:   0%|          | 0.00/138k [00:00<?, ?B/s]

train-00050.parquet:   0%|          | 0.00/139k [00:00<?, ?B/s]

train-00070.parquet:   0%|          | 0.00/139k [00:00<?, ?B/s]

...

train-00570.parquet:   0%|          | 0.00/141k [00:00<?, ?B/s]

train-00580.parquet:   0%|          | 0.00/143k [00:00<?, ?B/s]

train-00590.parquet:   0%|          | 0.00/139k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9600 [00:00<?, ? examples/s]

Dataset size: 9600

Sample formatted example:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 24 Jun 2026

You are the AI assistant for an airport counter-drone threat-detection system. Operators, responders, and members of the public ask you what is happening — sometimes formally, sometimes anxiously, sometimes casually — and you give them a clear, faithful answer based on the current sensor readings.

THE SENSOR NETWORK
The site is covered by 23 sensors arranged in a ring around the airfield, indexed 0 through 22. Each sensor carries three independent detectors:
- Radio Frequency (RF): classifies radio emissions as friendly or threat.
- Audio: classifies sound as Mambo drone (threat), Bebop drone (threat), or Background noise (no threat).
- Visual: a camera score from 0 (no drone visible) to 1 (drone clearly visible).

A fusion model combines every sensor's readings into a single system-wide threat estimate. You receive t

In [ ]:
sample_tokens = tokenizer(dataset[0]["text"], return_tensors="pt")
print(f"Token length of first example: {sample_tokens['input_ids'].shape[1]}")

Token length of first example: 2694


## Cell 6 — Train with SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

# ── Training config ───────────────────────────────────────
# num_train_epochs=3: 3 passes through the dummy dataset
#   → enough to confirm the pipeline works, not overfitting
# per_device_train_batch_size=2: small batch to fit T4 VRAM
# gradient_accumulation_steps=4: simulates batch size of 8
# warmup_steps=5: gradually ramp up learning rate at the start
# learning_rate=2e-4: standard for LoRA fine-tuning
# fp16=True: mixed precision on T4 (T4 doesn't support bf16)

training_args = SFTConfig(
    output_dir                  = "./llama-drone-adapter",
    num_train_epochs            = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    warmup_steps                = 10,        # was 5, increase for bigger dataset
    learning_rate               = 2e-4,
    fp16                        = False,
    bf16                        = True,
    logging_steps               = 20,        # was 5, log less often with 200 examples
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "linear",
    max_grad_norm               = 0.3,       # add this back as safety
    seed                        = 42,
    dataset_text_field          = "text",
    max_seq_length              = MAX_SEQ_LENGTH,
    push_to_hub=True,
    hub_model_id="JamesResearch1216/llama-drone-adapter",

    save_strategy="steps",       # Tells the trainer to save based on steps
    save_steps=50,              # Adjust this: saves and pushes every X steps
    hub_strategy="checkpoint",   # Pushes each checkpoint folder to your Hub repo
    save_total_limit=5,          # Keeps only the last 2 checkpoints locally to save disk space
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = dataset,
    args            = training_args,
)

print("Starting training...")
print(f"Training on {len(dataset)} examples for {training_args.num_train_epochs} epochs")
trainer_stats = trainer.train()
print("\nTraining complete!")
print(f"Train runtime: {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/9600 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training...
Training on 9600 examples for 1 epochs


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,600 | Num Epochs = 1 | Total steps = 600
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 5,636,096 of 1,241,450,496 (0.45% trained)


Step,Training Loss
20,0.684133
40,0.203538
60,0.181955
80,0.175227
100,0.168221
120,0.160775
140,0.160281
160,0.158302
180,0.156885
200,0.155663


Unsloth: Restored added_tokens_decoder metadata in ./llama-drone-adapter/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama-drone-adapter/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama-drone-adapter/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama-drone-adapter/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama-drone-adapter/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./llama-drone-adapter/checkpoint-250/tokenizer_config.json.


KeyboardInterrupt: 

## Cell 7 — Save the LoRA adapter

In [ ]:
# Save just the LoRA adapter (small ~50MB file, NOT the full 1B model)
# This is what gets swapped into the LangGraph agent later

SAVE_PATH = "./ARL/llama_drone_adapter"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"LoRA adapter saved to: {SAVE_PATH}")

# List what was saved
import os
for f in os.listdir(SAVE_PATH):
    size_mb = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

Unsloth: Restored added_tokens_decoder metadata in ./ARL/llama_drone_adapter/tokenizer_config.json.


LoRA adapter saved to: ./ARL/llama_drone_adapter
  adapter_config.json  (0.0 MB)
  adapter_model.safetensors  (22.6 MB)
  tokenizer.json  (17.2 MB)
  chat_template.jinja  (0.0 MB)
  README.md  (0.0 MB)
  tokenizer_config.json  (0.1 MB)


## Cell 8 — Quick sanity check (inference test)

In [ ]:
# Confirm the model can generate a response after fine-tuning
# This is just a smoke test — not a real evaluation

FastLanguageModel.for_inference(model)  # switch to faster inference mode

test_prompt = ALPACA_PROMPT.format(
    instruction="Is there a drone detected right now?",
    response=""  # leave blank — model fills this in
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens  = 128,
    temperature     = 0.7,
    do_sample       = True,
    pad_token_id    = tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=" * 60)
print("PROMPT:", test_prompt.strip())
print("-" * 60)
print("MODEL OUTPUT:")
# Extract just the Response part
if "### Response:" in response:
    print(response.split("### Response:")[-1].strip())
else:
    print(response)

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


PROMPT: You are an RF engineering assistant in the US army. You have access to three tools:
1. `check_current_rf_status`: Analyzes and provides sensors where anomalous rf transmission were detected.
2. `check_current_audio_status`: Checks audio signals to detect whether a drone is present. Use this whenever the user asks about audio, acoustics, or drone presence.
3. `check_current_visual_status`: Takes a picture/photo of the current environment using a camera, and identifies whether a drone is present or not.
4. `get_aggregate`: Populates the entire snapshot and uses the trained logistic regression model to make a prediction of whether a threat is present or not by aggregating all of the three data streams.
Based on the results from these tools, your task is to answer the questions so that even a civilian would understand what is going on.

### Instruction:
Is there a drone detected right now?

### Response:
------------------------------------------------------------
MODEL OUTPUT:
[{"

## Cell 9 — (Optional) Save to Google Drive

Skip if on Kaggle — just download the adapter folder manually.

In [ ]:
# Uncomment if running on Colab with Drive mounted

# import shutil
# drive_path = '/content/drive/MyDrive/ARL/llama_drone_adapter'
# shutil.copytree(SAVE_PATH, drive_path, dirs_exist_ok=True)
# print(f'Saved to Drive: {drive_path}')

---
## When teammates finish — swap in real data

Replace Cell 5's `dummy_examples` list with real examples built from `snapshot_log.csv`:

```python
import pandas as pd

df = pd.read_csv('./ARL/snapshot_log.csv')

real_examples = []
for _, row in df.iterrows():
    instruction = f"RF confidence: {row['rf_confidence']:.2f}, audio mambo: {row['audio_mambo']:.2f}, bebop: {row['audio_bebop']:.2f}, visual: {row['visual_confidence']:.2f}. What is the threat assessment?"
    label = 'THREAT DETECTED' if row['is_threat_gt'] == 1 else 'ZONE CLEAR'
    response = f"Based on the sensor readings, the multimodal fusion model classifies this as {label}. RF {'shows elevated activity' if row['rf_confidence'] > 0.5 else 'is within normal range'}. Audio classification {'indicates drone presence' if max(row['audio_mambo'], row['audio_bebop']) > 0.5 else 'shows background noise'}. Visual {'confirms drone' if row['visual_confidence'] > 0.5 else 'shows no visual threat'}."
    real_examples.append({"instruction": instruction, "response": response})

print(f"Built {len(real_examples)} real training examples from snapshot_log.csv")
```

Then re-run Cells 6 and 7 to retrain and save the updated adapter.